# RF Forge Playbook

**AI-powered Robot Framework test generation** — no terminal needed.

---

## Table of Contents

| # | Section | What it does |
|---|---|---|
| 0 | [Load .env](#setup--load-env) | Auto-loads AWS Bedrock credentials |
| 1 | [Setup — imports](#setup--imports-and-credential-check) | Imports RF Forge, verifies credentials |
| 2 | [Select skill](#step-1--select-skill) | Pick which skill to run from dropdown |
| 3 | [Configure session](#step-2--configure-session) | Instruction, codebase path, model, budget |
| 4 | [Verify values](#optional--verify-values) | Cross-check before running |
| 5 | [Run skill](#step-3--run-skill) | Execute the selected skill |
| 6 | [Session summary](#session-summary) | Cost + duration |

> **Tip:** JupyterLab sidebar TOC — click the **list icon** (three lines with dots) in the left sidebar.

---

## What this notebook does

Runs any of the 8 RF Forge utility skills with a form-based interface:

| Skill | What it generates |
|---|---|
| `create-account` | Account test cases (payload JSON, env file, .robot, README) |
| `import-pipeline` | Pipeline import .robot test cases |
| `upload-file` | File upload .robot test cases |
| `create-triggered-task` | Triggered task .robot test cases |
| `compare-csv` | CSV comparison .robot test cases |
| `verify-data-in-db` | Database verification .robot test cases |
| `export-data-to-csv` | Data export .robot test cases |
| `end-to-end-pipeline-verification` | Complete E2E .robot test suite |

## How to use

1. Run cells **top to bottom** (`Shift+Enter` in each cell)
2. Select a skill from the dropdown
3. Fill in the form (instruction, codebase path)
4. Click the **Run** button — wait 1-5 minutes
5. Check your codebase for generated files

## Prerequisites

- RF Forge installed: `pip install -e ai/` (from the project root)
- The `.env` file is auto-loaded by the first cell — no need to source it before launching

---
## Setup — Load .env

In [ ]:
import os
from pathlib import Path

# ──────────────────────────────────────────────────────────
# Auto-load .env so you don't need to source it before launching Jupyter
# ──────────────────────────────────────────────────────────
def _load_dotenv(env_path):
    """Read a .env file and set os.environ for each KEY=VALUE line."""
    if not env_path.exists():
        print(f'⚠️  .env file not found at {env_path}')
        return 0
    count = 0
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, _, value = line.partition('=')
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ[key] = value
            count += 1
    return count

# Walk up from this notebook to find the project root .env
# notebooks/ → ai/ → {{cookiecutter}}/ (where .env lives)
_notebook_dir = Path(os.getcwd())
for _candidate in [_notebook_dir, _notebook_dir.parent, _notebook_dir.parent.parent, _notebook_dir.parent.parent.parent]:
    if (_candidate / '.env').exists():
        _env_file = _candidate / '.env'
        break
else:
    _env_file = _notebook_dir / '.env'

loaded = _load_dotenv(_env_file)
if loaded:
    print(f'✅ Loaded {loaded} vars from {_env_file}')
else:
    print(f'⚠️  No vars loaded from {_env_file}')

---
## Setup — imports and credential check

Run this cell after cell 1 (which loads `.env`). It imports RF Forge, patches asyncio for Jupyter, and verifies your Bedrock credentials are set.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

# Allow asyncio.run() inside Jupyter's event loop
import nest_asyncio
nest_asyncio.apply()

# Import RF Forge
from rf_forge.skills import SKILLS
from rf_forge.runner import run_skill

# UI widgets
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

# Sanity-check Bedrock env vars (loaded by cell 1)
required = ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_REGION', 'ANTHROPIC_MODEL', 'CLAUDE_CODE_USE_BEDROCK']
missing = [v for v in required if not os.getenv(v)]

if missing:
    print(f'⚠️  Missing env vars: {missing}')
    print('Run cell 1 first (the _load_dotenv cell), or check your .env file.')
else:
    print('✅ Bedrock credentials loaded')
    print(f'   Model:  {os.getenv("ANTHROPIC_MODEL")}')
    print(f'   Region: {os.getenv("AWS_REGION")}')
    print(f'✅ {len(SKILLS)} skills available: {", ".join(SKILLS.keys())}')

---
## Step 1 — Select skill

Pick which utility skill to run. Each skill generates specific Robot Framework test artifacts.

In [ ]:
# Skill dropdown — maps display name to skill key
skill_options = [
    ('Create Account — payload JSON, env, .robot, README', 'create-account'),
    ('Import Pipeline — import .slp pipeline files', 'import-pipeline'),
    ('Upload File — upload JSON, CSV, JARs to SLDB', 'upload-file'),
    ('Create Triggered Task — create + execute tasks', 'create-triggered-task'),
    ('Compare CSV — actual vs expected output', 'compare-csv'),
    ('Verify Data in DB — row counts, data integrity', 'verify-data-in-db'),
    ('Export Data to CSV — database table → CSV', 'export-data-to-csv'),
    ('End-to-End Pipeline — complete E2E test suite', 'end-to-end-pipeline-verification'),
]

skill_w = widgets.Dropdown(
    options=skill_options,
    value='create-account',
    description='Skill:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='90%'),
)

# Show skill description when selection changes
skill_info = widgets.Output()

def on_skill_change(change):
    with skill_info:
        clear_output()
        skill = SKILLS[change['new']]
        print(f'📋 {skill.name}: {skill.description}')

skill_w.observe(on_skill_change, names='value')

display(skill_w, skill_info)

# Trigger initial display
on_skill_change({'new': skill_w.value})

---
## Step 2 — Configure session

**Run this cell** (`Shift+Enter`) and fill in the form. Values are preserved across re-runs.

| Field | What to enter |
|---|---|
| **File path** | *(Optional)* Path to a .md file with your instruction |
| **Instruction** | What to generate — e.g., "Create Oracle and Snowflake accounts" |
| **Codebase path** | Path to the RF project root (where .robot files will be created) |
| **Model** | Sonnet (daily use), Opus (best quality), Haiku (cheapest) |
| **Budget** | Dollar cap per run |

In [ ]:
# Defaults — detect RF project root automatically
_nb_dir = Path(os.getcwd())
_default_codebase = str(_nb_dir) if (_nb_dir / 'test').is_dir() else str(_nb_dir.parent.parent) if (_nb_dir.parent.parent / 'test').is_dir() else str(_nb_dir)
DEFAULT_INSTRUCTION = 'Create Oracle and Snowflake accounts'

# ── Preserve values if widgets already exist (re-run safe) ──
_prev = {}
try:
    _prev['req_file'] = req_file_w.value
    _prev['instruction'] = instruction_w.value
    _prev['codebase'] = codebase_w.value
    _prev['model'] = model_w.value
    _prev['budget'] = budget_w.value
except NameError:
    pass

# ── Load from file (optional) ──────────────────────────
file_label = widgets.HTML(
    value='<b>📄 Load from file</b> <span style="color:gray">'
          '(optional — loads into Instruction box below. You can then edit to add your own notes.)</span>'
)

req_file_w = widgets.Text(
    value=_prev.get('req_file', ''),
    placeholder='Path to a .md file — e.g., ./requirements/JIRA-4567.md',
    description='File path:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '120px'},
)

load_file_btn = widgets.Button(
    description='📄 Load file',
    button_style='info',
    layout=widgets.Layout(width='150px', height='32px'),
)
load_file_output = widgets.Output()

def on_load_file(b):
    with load_file_output:
        clear_output()
        file_path = Path(req_file_w.value.strip())
        if not req_file_w.value.strip():
            print('ℹ️  Enter a file path first.')
            return
        if not file_path.exists():
            print(f'❌ File not found: {file_path}')
            return
        content = file_path.read_text()
        instruction_w.value = content
        print(f'✅ Loaded {len(content)} chars from {file_path.name} → edit the Instruction box below to add notes')

load_file_btn.on_click(on_load_file)

separator = widgets.HTML(value='<hr style="margin:8px 0; border:0; border-top:1px solid #ddd;">')

# ── Instruction ─────────────────────────────────────────
instr_label = widgets.HTML(
    value='<b>Instruction</b> <span style="color:gray">'
          '(describe what to generate — or load a file above then edit here)</span>'
)

instruction_w = widgets.Textarea(
    value=_prev.get('instruction', DEFAULT_INSTRUCTION),
    placeholder='e.g., Create Oracle and Snowflake accounts for pipeline testing',
    layout=widgets.Layout(width='90%', height='120px'),
)

# ── Other fields ────────────────────────────────────────
codebase_w = widgets.Text(
    value=_prev.get('codebase', _default_codebase),
    description='Codebase path:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '120px'},
)

model_w = widgets.Dropdown(
    options=[('Sonnet (recommended)', 'sonnet'), ('Opus (best quality)', 'opus'), ('Haiku (cheapest)', 'haiku')],
    value=_prev.get('model', 'sonnet'),
    description='Model:',
    style={'description_width': '120px'},
)

budget_w = widgets.FloatSlider(
    value=_prev.get('budget', 1.0),
    min=0.1, max=5.0, step=0.1,
    description='Budget (USD):',
    readout_format='.1f',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%'),
)

# ── Reset button ────────────────────────────────────────
reset_btn = widgets.Button(
    description='🔄 Reset to defaults',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='32px'),
)

def on_reset(b):
    req_file_w.value = ''
    instruction_w.value = DEFAULT_INSTRUCTION
    codebase_w.value = _default_codebase
    model_w.value = 'sonnet'
    budget_w.value = 1.0
    with load_file_output:
        clear_output()
    print('🔄 All fields reset to defaults')

reset_btn.on_click(on_reset)

display(widgets.VBox([
    file_label,
    req_file_w,
    widgets.HBox([load_file_btn, load_file_output]),
    separator,
    instr_label,
    instruction_w,
    codebase_w,
    model_w,
    budget_w,
    widgets.HTML(value='<hr style="margin:8px 0; border:0; border-top:1px solid #ddd;">'),
    reset_btn,
]))

---
## Optional — Verify values

Run the cell below to cross-check all form values before running. Catches typos in paths before you spend money.

In [ ]:
# ── Optional: verify your form values before running ──
print('📋 Current values:\n')
print(f'   Skill:          {skill_w.value}')
print(f'   Instruction:    {instruction_w.value[:100]}{"..." if len(instruction_w.value) > 100 else ""}')
print(f'   Req file:       {req_file_w.value or "(not set)"}')
print(f'   Codebase path:  {codebase_w.value}')
print(f'   Model:          {model_w.value}')
print(f'   Budget:         ${budget_w.value:.2f}')

# ── Validate req file path (if entered) ──
print('\n── Path validation ──')
if req_file_w.value.strip():
    req_file_path = Path(req_file_w.value.strip())
    if req_file_path.exists():
        if req_file_path.is_file():
            size_kb = req_file_path.stat().st_size / 1024
            print(f'✅ Req file exists ({req_file_path.name}, {size_kb:.1f} KB)')
        else:
            print(f'⚠️  Req file path is a folder, not a file: {req_file_path}')
    else:
        print(f'❌ Req file does NOT exist: {req_file_w.value}')
else:
    print(f'ℹ️  Req file: not set (using Instruction text directly)')

# ── Validate codebase path ──
codebase_path = Path(codebase_w.value)
if codebase_path.exists():
    if codebase_path.is_dir():
        has_test = (codebase_path / 'test').is_dir()
        has_claude = (codebase_path / '.claude' / 'skills').is_dir()
        items = len(list(codebase_path.iterdir()))
        print(f'✅ Codebase path exists (folder with {items} items)')
        if has_test:
            print(f'   ✅ test/ folder found')
        else:
            print(f'   ⚠️  No test/ folder — are you pointing at the right project root?')
        if has_claude:
            print(f'   ✅ .claude/skills/ found ({len(list((codebase_path / ".claude" / "skills").iterdir()))} skills)')
    else:
        print(f'✅ Codebase path exists (single file: {codebase_path.name})')
else:
    print(f'❌ Codebase path does NOT exist: {codebase_w.value}')

print(f'\n💡 If values look wrong, go back to the form cells and edit them. Re-run this cell to verify again.')

---
## Step 3 — Run skill

Click the button below to run the selected skill. The button greys out while running and re-enables when done.

Files are generated directly in your codebase (the `--codebase-path` folder).

In [ ]:
run_btn = widgets.Button(
    description='▶️  Run selected skill',
    button_style='primary',
    layout=widgets.Layout(width='300px', height='45px'),
)
run_output = widgets.Output()

# Store stats for the session summary cell
_last_stats = {}

def on_run_skill(b):
    global _last_stats

    # Disable button while running
    b.disabled = True
    original_desc = f'▶️  Run {skill_w.value}'
    b.description = '⏳ Running...'
    b.button_style = ''

    with run_output:
        clear_output()

        # Validate codebase path
        codebase = codebase_w.value.strip()
        if not Path(codebase).exists():
            print(f'❌ Codebase path does not exist: {codebase}')
            b.disabled = False
            b.description = original_desc
            b.button_style = 'primary'
            return

        # Get the skill
        skill = SKILLS[skill_w.value]

        print(f'▶️  Running {skill.name}...')
        print(f'   Codebase:  {codebase}')
        print(f'   Model:     {model_w.value}')
        print(f'   Budget:    ${budget_w.value:.2f}\n')

        try:
            stats = run_skill(
                skill=skill,
                args={
                    'instruction': instruction_w.value,
                    'codebase_path': codebase,
                },
                model=model_w.value,
                max_budget=budget_w.value,
                cwd=codebase,
            )
            if stats:
                _last_stats = stats
                _last_stats['skill'] = skill.name
                _last_stats['codebase'] = codebase
                print(f'\n✅ Done. Stats: {stats}')
            else:
                print(f'\n⚠️  Completed with no stats returned.')
        except Exception as e:
            print(f'❌ Error: {e}')

    # Re-enable button
    b.disabled = False
    b.description = f'▶️  Run {skill_w.value}'
    b.button_style = 'primary'

# Update button label when skill changes
def on_skill_btn_update(change):
    run_btn.description = f'▶️  Run {change["new"]}'

skill_w.observe(on_skill_btn_update, names='value')
run_btn.description = f'▶️  Run {skill_w.value}'

run_btn.on_click(on_run_skill)
display(run_btn, run_output)

---
## Session summary

Run this cell after the skill completes to see cost and duration.

In [ ]:
if _last_stats:
    print(f'📊 Last run summary:\n')
    print(f'   Skill:     {_last_stats.get("skill", "unknown")}')
    print(f'   Turns:     {_last_stats["num_turns"]}')
    print(f'   Cost:      ${_last_stats["cost_usd"]:.4f}')
    print(f'   Duration:  {_last_stats["duration_ms"] / 1000:.1f}s')
    print(f'   Codebase:  {_last_stats.get("codebase", "unknown")}')
    print(f'\n💡 Check your codebase for generated files:')
    print(f'   ls {_last_stats.get("codebase", ".")}/test/suite/pipeline_tests/')
else:
    print('ℹ️  No skill has been run yet. Run Step 3 first.')